# Asad Nabi Bakhsh (SP22-BCS-025)
# Shahmeer Hamza (SP22-BCS-039)

 Semester Project : Soulmate Matcher

 Submitted To : Sir Ehtisham ul Huque

Date : 16 Dec 2025

In [1]:
# Step 1: Install dependencies
!pip install facenet-pytorch mpi4py cupy-cuda12x
!nvidia-smi

Tue Dec 16 14:25:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install --upgrade pip
!pip install pillow==9.5.0
!pip install facenet-pytorch mpi4py cupy-cuda12x

In [10]:
!pip uninstall -y pillow facenet-pytorch

Found existing installation: Pillow 9.5.0
Uninstalling Pillow-9.5.0:
  Successfully uninstalled Pillow-9.5.0
Found existing installation: facenet-pytorch 2.5.2
Uninstalling facenet-pytorch-2.5.2:
  Successfully uninstalled facenet-pytorch-2.5.2


In [3]:
!pip install pillow==9.5.0
!pip install facenet-pytorch==2.5.2
!pip install mpi4py cupy-cuda12x


# Soulmate Face Matcher

In [63]:
%%writefile matcher.cu
#include <iostream>
#include <vector>
#include <cmath>
#include <cstdio>
#include <mpi.h>
#include <cuda_runtime.h>

// --- CUDA KERNEL ---
__global__ void distanceKernel(float* query, float* db_vecs, float* results, int dim, int num_local_imgs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Debug: Force write to prove kernel is running
    if (idx < num_local_imgs) {
        float sum = 0.0f;
        float* current_db = &db_vecs[idx * dim];
        for (int i = 0; i < dim; ++i) {
            float diff = current_db[i] - query[i];
            sum += diff * diff;
        }
        results[idx] = sqrt(sum);
    }
}

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);
    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int num_db_images = 0;
    int dim = 0;
    std::vector<float> query_vec;
    std::vector<float> all_db_vecs;

    //  READ DATA ---
    if (rank == 0) {
        FILE* f = fopen("temp_bridge.bin", "rb");
        if (!f) { printf("Error opening file\n"); MPI_Abort(MPI_COMM_WORLD, 1); }

        fread(&num_db_images, sizeof(int), 1, f);
        fread(&dim, sizeof(int), 1, f);

        query_vec.resize(dim);
        all_db_vecs.resize(num_db_images * dim);

        fread(query_vec.data(), sizeof(float), dim, f);
        fread(all_db_vecs.data(), sizeof(float), num_db_images * dim, f);
        fclose(f);

        // --- CPU CALCULATION CHECK ---
        // We calculate distance on CPU to prove data integrity
        float cpu_sum = 0.0f;
        for(int i=0; i<dim; i++) {
            float d = all_db_vecs[i] - query_vec[i];
            cpu_sum += d*d;
        }
        printf("[CPU CHECK] Rank 0 CPU Calculated Distance: %.4f\n", sqrt(cpu_sum));
    }

    // Broadcast Dimensions
    MPI_Bcast(&num_db_images, 1, MPI_INT, 0, MPI_COMM_WORLD);
    MPI_Bcast(&dim, 1, MPI_INT, 0, MPI_COMM_WORLD);

    // Resize & Broadcast Vectors
    if (rank != 0) query_vec.resize(dim);
    MPI_Bcast(query_vec.data(), dim, MPI_FLOAT, 0, MPI_COMM_WORLD);

    if (rank != 0) all_db_vecs.resize(num_db_images * dim);
    MPI_Bcast(all_db_vecs.data(), num_db_images * dim, MPI_FLOAT, 0, MPI_COMM_WORLD);

    // --- GPU SETUP ---
    // Only Rank 0 works for this specific 1-image test to avoid complexity
    int local_n = (rank == 0) ? num_db_images : 0;

    // Initialize with -1.0 to detect kernel failure
    std::vector<float> local_results(local_n, -1.0f);

    if (local_n > 0) {
        float *d_query, *d_db, *d_res;

        // Allocate
        cudaMalloc(&d_query, dim * sizeof(float));
        cudaMalloc(&d_db, local_n * dim * sizeof(float));
        cudaMalloc(&d_res, local_n * sizeof(float));

        // Copy Data
        cudaMemcpy(d_query, query_vec.data(), dim * sizeof(float), cudaMemcpyHostToDevice);
        cudaMemcpy(d_db, all_db_vecs.data(), local_n * dim * sizeof(float), cudaMemcpyHostToDevice);

        // Initialize Device Result to -5.0 to detect if kernel writes nothing
        std::vector<float> init_res(local_n, -5.0f);
        cudaMemcpy(d_res, init_res.data(), local_n * sizeof(float), cudaMemcpyHostToDevice);

        // LAUNCH KERNEL
        distanceKernel<<<1, 256>>>(d_query, d_db, d_res, dim, local_n);
        cudaDeviceSynchronize();

        // Check for Kernel Errors
        cudaError_t err = cudaGetLastError();
        if (err != cudaSuccess) {
            printf("[GPU ERROR] Kernel Launch Failed: %s\n", cudaGetErrorString(err));
        }

        // Read Back
        cudaMemcpy(local_results.data(), d_res, local_n * sizeof(float), cudaMemcpyDeviceToHost);

        cudaFree(d_query); cudaFree(d_db); cudaFree(d_res);

        // --- FINAL REPORT ---
        printf("\n[GPU RESULT] Rank %d calculated: %.4f\n", rank, local_results[0]);

        if (local_results[0] < 0) {
            printf(">> ERROR: Kernel did not execute (Result is negative)\n");
        } else if (local_results[0] < 1.0) {
            printf(">> MATCH: Likely SAME person\n");
        } else {
            printf(">> NO MATCH: Different person\n");
        }
    }

    MPI_Finalize();
    return 0;
}

Overwriting matcher.cu


In [64]:
# Install MPI library
!apt-get install openmpi-bin openmpi-common libopenmpi-dev > /dev/null
import os
import subprocess

# Detect GPU Architecture
try:
    # Query NVIDIA-SMI for the GPU name
    result = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'])
    gpu_name = result.decode('utf-8').strip()
    print(f"Detected GPU: {gpu_name}")

    # Map GPU to Architecture Flag
    if "T4" in gpu_name:
        arch_flag = "-arch=sm_75"
    elif "P100" in gpu_name:
        arch_flag = "-arch=sm_60"
    elif "V100" in gpu_name:
        arch_flag = "-arch=sm_70"
    elif "A100" in gpu_name:
        arch_flag = "-arch=sm_80"
    elif "L4" in gpu_name:
        arch_flag = "-arch=sm_89"
    else:
        # Fallback for older/unknown GPUs
        arch_flag = "-arch=sm_50"

    print(f"Applying Compiler Flag: {arch_flag}")

except Exception as e:
    print("Could not detect GPU automatically. Defaulting to T4 (sm_75).")
    arch_flag = "-arch=sm_75"

# Compile with the specific Architecture Flag
compile_cmd = (
    f"nvcc {arch_flag} -I/usr/lib/x86_64-linux-gnu/openmpi/include "
    f"-L/usr/lib/x86_64-linux-gnu/openmpi/lib "
    f"-lmpi matcher.cu -o matcher"
)

print(f"Running: {compile_cmd}")
os.system(compile_cmd)
print("Compilation Complete!")
print("Compilation successful! 'matcher' executable created.")

Detected GPU: Tesla T4
Applying Compiler Flag: -arch=sm_75
Running: nvcc -arch=sm_75 -I/usr/lib/x86_64-linux-gnu/openmpi/include -L/usr/lib/x86_64-linux-gnu/openmpi/lib -lmpi matcher.cu -o matcher
Compilation Complete!
Compilation successful! 'matcher' executable created.


In [ ]:
import gradio as gr
import torch
from facenet_pytorch import InceptionResnetV1
from PIL import Image
import numpy as np
import subprocess
import struct
import os
import re

# --- 1. SETUP BACKEND ---
# Load model once at startup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading Model on {device}...")
model = InceptionResnetV1(pretrained='vggface2').eval().to(device)
print("Model Loaded!")

# --- 2. CORE LOGIC FUNCTION ---
def run_face_matching(image_a, image_b):
    if image_a is None or image_b is None:
        return "⚠️ Error: Please upload both images."

    try:
        # A. PREPROCESS IMAGES
        # Gradio provides images as numpy arrays, convert to PIL
        img1 = Image.fromarray(image_a).convert('RGB').resize((160, 160))
        img2 = Image.fromarray(image_b).convert('RGB').resize((160, 160))

        # B. GENERATE EMBEDDINGS (Python)
        embeddings = []
        for img in [img1, img2]:
            img_tensor = torch.tensor(np.array(img)).permute(2, 0, 1).float() / 255.0
            with torch.no_grad():
                emb = model(img_tensor.unsqueeze(0).to(device)).cpu().numpy().flatten()
                embeddings.append(emb)

        query_emb = embeddings[0].astype(np.float32)
        db_emb = embeddings[1].astype(np.float32)
        dim = query_emb.shape[0]

        # C. BRIDGE TO C++ (Write Binary)
        bridge_file = "ui_bridge.bin"
        with open(bridge_file, "wb") as f:
            f.write(struct.pack('ii', 1, dim))
            f.write(query_emb.tobytes())
            f.write(db_emb.tobytes())

        # D. EXECUTE C++ ENGINE
        # Make sure 'matcher' exists from previous steps
        if not os.path.exists("./matcher"):
            return " Error: C++ 'matcher' executable not found. Please compile it first."

        my_env = os.environ.copy()
        my_env["LD_LIBRARY_PATH"] = "/usr/lib/x86_64-linux-gnu/openmpi/lib:" + my_env.get("LD_LIBRARY_PATH", "")

        # Note: Using bridge_file name in C++ requires C++ change or renaming file
        # Hack: Rename our UI file to the one C++ expects ('temp_bridge.bin')
        if os.path.exists("temp_bridge.bin"): os.remove("temp_bridge.bin")
        os.rename(bridge_file, "temp_bridge.bin")

        process = subprocess.Popen(
            ['mpirun', '--allow-run-as-root', '--oversubscribe', '-np', '2', './matcher'],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            env=my_env
        )

        stdout, stderr = process.communicate()
        raw_output = stdout.decode('utf-8')

        # PARSE C++ OUTPUT FOR UI
        # We look for the specific line: "[GPU RESULT] Rank 0 calculated: X.XXXX"
        match = re.search(r"Rank 0 calculated: ([0-9.]+)", raw_output)

        result_text = ""
        if match:
            dist = float(match.group(1))

            # Create a nice UI Output
            if dist < 1.0:
                result_text = f" **MATCH FOUND!**\n\nDistance: {dist:.4f}\nConfidence: High\nStatus: Same Person"
            else:
                result_text = f" **NO MATCH**\n\nDistance: {dist:.4f}\nConfidence: High\nStatus: Different People"

            # Append Debug info at bottom slightly smaller
            result_text += f"\n\n--- C++ Backend Log ---\n{raw_output}"
        else:
            result_text = f" **Backend Error**\nCould not parse C++ output.\nRaw Output:\n{raw_output}\nError Log:\n{stderr.decode('utf-8')}"

        return result_text

    except Exception as e:
        return f"System Error: {str(e)}"

# BUILD UI INTERFACE ---
with gr.Blocks(title="CUDA Face Matcher") as app:
    gr.Markdown("# Asad's Face Matching System")
    gr.Markdown("Hybrid Python (Inference) + C++ CUDA (Comparison) Architecture")

    with gr.Row():
        with gr.Column():
            img_input1 = gr.Image(label="Image 1(Person A)")
        with gr.Column():
            img_input2 = gr.Image(label="Image 2(Person B)")

    btn = gr.Button("Run Soulmate Matcher", variant="primary")

    output_text = gr.Textbox(label="Matching Result", lines=10)

    btn.click(fn=run_face_matching, inputs=[img_input1, img_input2], outputs=output_text)

# 4. LAUNCH ---
print("Launching UI...")
app.launch(debug=True, share=True)

Loading Model on cuda...
Model Loaded!
Launching UI...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d2b42bb6680ae7404d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
